In [ ]:
# =============================
# 0) Setup & Imports
# =============================
!pip install torch torchvision

import os, math, random
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, utils as vutils
from torchvision.datasets.folder import default_loader
from google.colab import drive
import matplotlib.pyplot as plt

# Mount Google Drive
drive.mount('/content/drive')

# =============================
# 1) Config
# =============================
class Config:
    data_root = "/content/drive/MyDrive/Stroke"   # <-- your folder with 500 images
    save_dir  = "/content/drive/MyDrive/facial_palsy_outputs"
    img_size  = 128
    channels  = 3
    latent_dim = 128
    g_features = 64
    d_features = 64
    batch_size = 64
    epochs = 800
    lr = 2e-4
    beta1 = 0.5
    beta2 = 0.999
    n_workers = 2
    sample_z = 64
    seed = 42
    device = "cuda" if torch.cuda.is_available() else "cpu"

cfg = Config()
os.makedirs(cfg.save_dir, exist_ok=True)
for sub in ["samples", "checkpoints", "generation"]:
    os.makedirs(os.path.join(cfg.save_dir, sub), exist_ok=True)

random.seed(cfg.seed)
torch.manual_seed(cfg.seed)

# =============================
# 2) Dataset Loader (Single Folder)
# =============================
transform = transforms.Compose([
    transforms.Resize(cfg.img_size),
    transforms.CenterCrop(cfg.img_size),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*cfg.channels, [0.5]*cfg.channels),
])

class SingleFolderDataset(Dataset):
    def __init__(self, root, transform=None):
        self.root = root
        self.transform = transform
        self.paths = [os.path.join(root, f) for f in os.listdir(root)
                      if f.lower().endswith((".jpg",".jpeg",".png"))]
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        img = default_loader(self.paths[idx])
        if self.transform:
            img = self.transform(img)
        return img, 0   # dummy label

dataset = SingleFolderDataset(cfg.data_root, transform=transform)
loader = DataLoader(dataset, batch_size=cfg.batch_size, shuffle=True,
                    num_workers=cfg.n_workers, pin_memory=True)

print("Total images found:", len(dataset))

# =============================
# 3) DCGAN Models
# =============================
class GenBlock(nn.Module):
    def __init__(self, in_c, out_c, k=4, s=2, p=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(in_c, out_c, k, s, p, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(True)
        )
    def forward(self, x): return self.net(x)

class Generator(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.main = nn.Sequential(
            GenBlock(cfg.latent_dim, cfg.g_features*16, 4, 1, 0), # 1->4
            GenBlock(cfg.g_features*16, cfg.g_features*8),        # 4->8
            GenBlock(cfg.g_features*8, cfg.g_features*4),         # 8->16
            GenBlock(cfg.g_features*4, cfg.g_features*2),         # 16->32
            GenBlock(cfg.g_features*2, cfg.g_features),           # 32->64
            nn.ConvTranspose2d(cfg.g_features, cfg.channels, 4, 2, 1, bias=False),
            nn.Tanh()
        )
    def forward(self, z): return self.main(z)

class DisBlock(nn.Module):
    def __init__(self, in_c, out_c, bn=True):
        super().__init__()
        layers = [nn.Conv2d(in_c, out_c, 4, 2, 1, bias=False)]
        if bn: layers.append(nn.BatchNorm2d(out_c))
        layers.append(nn.LeakyReLU(0.2, inplace=True))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

class Discriminator(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(cfg.channels, cfg.d_features, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            DisBlock(cfg.d_features, cfg.d_features*2),
            DisBlock(cfg.d_features*2, cfg.d_features*4),
            DisBlock(cfg.d_features*4, cfg.d_features*8),
            nn.Conv2d(cfg.d_features*8, 1, 4, 1, 0, bias=False)
        )
    def forward(self, x): return self.main(x).view(-1)

def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

# =============================
# 4) Initialize Models
# =============================
G = Generator(cfg).to(cfg.device)
D = Discriminator(cfg).to(cfg.device)
G.apply(weights_init)
D.apply(weights_init)

optG = torch.optim.Adam(G.parameters(), lr=cfg.lr, betas=(cfg.beta1, cfg.beta2))
optD = torch.optim.Adam(D.parameters(), lr=cfg.lr, betas=(cfg.beta1, cfg.beta2))
criterion = nn.BCEWithLogitsLoss()

fixed_z = torch.randn(cfg.sample_z, cfg.latent_dim, 1, 1, device=cfg.device)

# =============================
# 5) Training Loop
# =============================
for epoch in range(1, cfg.epochs+1):
    for i, (real, _) in enumerate(loader):
        real = real.to(cfg.device)
        b = real.size(0)

        # Train D
        z = torch.randn(b, cfg.latent_dim, 1, 1, device=cfg.device)
        fake = G(z).detach()
        D_real = D(real)
        D_fake = D(fake)
        lossD = criterion(D_real, torch.ones_like(D_real)) + criterion(D_fake, torch.zeros_like(D_fake))
        optD.zero_grad(); lossD.backward(); optD.step()

        # Train G
        z = torch.randn(b, cfg.latent_dim, 1, 1, device=cfg.device)
        fake = G(z)
        D_fake = D(fake)
        lossG = criterion(D_fake, torch.ones_like(D_fake))
        optG.zero_grad(); lossG.backward(); optG.step()

        if i % 20 == 0:
            print(f"[Epoch {epoch}/{cfg.epochs}] [Batch {i}/{len(loader)}] Loss D: {lossD.item():.4f}, Loss G: {lossG.item():.4f}")

# =============================
# 6) Generate Final Images + Save Individually
# =============================
G.eval()
with torch.no_grad():
    z = torch.randn(64, cfg.latent_dim, 1, 1, device=cfg.device)
    gen_imgs = G(z).cpu()

# Make a folder for individual images
indiv_dir = os.path.join(cfg.save_dir, "generation/individual")
os.makedirs(indiv_dir, exist_ok=True)

# Save each image separately
for idx, img in enumerate(gen_imgs):
    vutils.save_image(img, f"{indiv_dir}/gen_{idx:03d}.png", normalize=True, value_range=(-1,1))

print(f"✅ {len(gen_imgs)} individual images saved to: {indiv_dir}")

# (Optional) still save a grid preview
grid = vutils.make_grid(gen_imgs, nrow=8, normalize=True, value_range=(-1,1))
out_path = f"{cfg.save_dir}/generation/final_generated_grid.png"
vutils.save_image(grid, out_path)

plt.figure(figsize=(8,8))
plt.axis("off")
plt.title("Generated Faces (Grid Preview)")
plt.imshow(grid.permute(1,2,0).numpy())
plt.show()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total images found: 2313
[Epoch 1/800] [Batch 0/37] Loss D: 1.5262, Loss G: 2.7519
[Epoch 1/800] [Batch 20/37] Loss D: 1.0632, Loss G: 4.9014
[Epoch 2/800] [Batch 0/37] Loss D: 0.9679, Loss G: 4.6506
[Epoch 2/800] [Batch 20/37] Loss D: 0.4435, Loss G: 4.5508
[Epoch 3/800] [Batch 0/37] Loss D: 0.3688, Loss G: 4.6996
[Epoch 3/800] [Batch 20/37] Loss D: 0.1835, Loss G: 4.6462
[Epoch 4/800] [Batch 0/37] Loss D: 0.1077, Loss G: 4.8109
[Epoch 4/800] [Batch 20/37] Loss D: 0.1074, Loss G: 5.0513
[Epoch 5/800] [Batch 0/37] Loss D: 0.1128, Loss G: 4.4736
[Epoch 5/800] [Batch 20/37] Loss D: 0.1628, Loss G: 5.3194
[Epoch 6/800] [Batch 0/37] Loss D: 0.1126, Loss G: 4.9544
[Epoch 6/800] [Batch 20/37] Loss D: 0.1033, Loss G: 5.5500
[Epoch 7/800] [Batch 0/37] Loss D: 0.0835, Loss G: 5.3889
[Epoch 7/800] [Batch 20/37] Loss D: 0.0553, Loss G: 5.3635
[Epoch 8/800] [Batch 0/37] 

In [ ]:
import matplotlib.pyplot as plt

# Suppose you store losses during training like this:
D_losses = []
G_losses = []

# inside your training loop:
# after computing each batch loss, append:
# D_losses.append(d_loss.item())
# G_losses.append(g_loss.item())

# ==========================
# After training, plot them:
# ==========================
plt.figure(figsize=(10,5))
plt.title("Generator and Discriminator Loss During Training")
plt.plot(G_losses, label="G (Generator)")
plt.plot(D_losses, label="D (Discriminator)")
plt.xlabel("Iterations")
plt.ylabel("Loss")
plt.legend()
plt.show()
